setup

In [1]:
import json
import sys
from collections import namedtuple
from dataclasses import dataclass
from pathlib import Path

import einops
import numpy as np
import torch as t
import torch.nn as nn
import torch.nn.functional as F
import torchinfo
from IPython.display import display
from jaxtyping import Float, Int
from PIL import Image
from rich import print as rprint
from rich.table import Table
from torch import Tensor
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm.notebook import tqdm

# Make sure exercises are in the path
chapter = "chapter0_fundamentals"
section = "part2_cnns"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

MAIN = __name__ == "__main__"

import part2_cnns.tests as tests
import part2_cnns.utils as utils
from plotly_utils import line

## 1️. Making your own modules

### Subclassing `nn.Module` (base class for all neural network components)
- `__init__`: create + setup model components like parameters, hyperparameters, and submodles
    - `super().__init__()` initializes the parent `nn.Module`, enabling parameter tracking and all core PyTorch functionality
- `forward`: define the computation and pass input through the model. we don't call it directly. 
- `nn.Parameter`: a special tensor that PyTorch knows should be trained/updated (learnable data).
- `extra_reper`: controls what gets printed when we do `print(model)`. Useful for debugging/inspecting.

### ReLU

The first module you should implement is `ReLU`. This will relatively simple, since it doesn't involve any argument (so we only need to implement `forward`). Make sure you look at the PyTorch documentation page for [ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html) so that you're comfortable with how they work.

ReLU is defined as the element-wise maximum between the input and a tensor of zeros. It's one of the simplest types of **nonlinear activation functions**. These are essential because linear operations compose to make more linear operations, which is very limiting. On the other hand, the **universal approximation theorem** tells us that we can approximate any continuous function using a sufficiently large neural network, if we use nonlinear activation functions. It's worth emphasizing that the theory of the UAT and what networks look like in practice are very different - in particular, many versions of the UAT are based on a shallow but extremely wide neural network, on the other hand most of the power of modern neural networks comes from their ability to compose between layers: feeding the output of one layer into the input of another, and create increasingly expressive functions. We'll explore this idea more when we study circuits in next week's interpretability material.

### Exercise - implement `ReLU`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> ```

You should fill in the `forward` method of the `ReLU` class below.

> `torch.maximum`: Element-wise comparison between two tensors

In [2]:
class ReLU(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        return t.maximum(t.tensor(0.0), x)

In [3]:
tests.test_relu(ReLU)

All tests in `test_relu` passed!


### Linear

Now implement your own `Linear` module. This applies a simple linear transformation, with a weight matrix and optional bias vector. The PyTorch documentation page is [here](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html). Note that this is the first `Module` you'll implement that has learnable weights and biases.

<details>
<summary>Question - what type do you think these variables should be?</summary>

They have to be `torch.Tensor` objects wrapped in `nn.Parameter` in order for `nn.Module` to recognize them. If you forget to do this, `module.parameters()` won't include your `Parameter`, which prevents an optimizer from being able to modify it during training.
        
Also, in tomorrow's exercises we'll be building a ResNet and loading in weights from a pretrained model, and this is hard to do if you haven't registered all your parameters!
</details>

For any layer, initialization is very important for the stability of training: with a bad initialization, your model will take much longer to converge or may completely fail to learn anything. The default PyTorch behavior isn't necessarily optimal and you can often improve performance by using something more custom, but we'll follow it for today because it's simple and works decently well.

Each float in the weight and bias tensors are drawn independently from the uniform distribution on the interval:

$$
\bigg[-\frac{1}{\sqrt{N_{in}}}, \frac{1}{\sqrt{N_{in}}}\bigg]
$$

where $N_{in}$ is the number of inputs contributing to each output value. The rough intuition for this is that it keeps the variance of the activations at each layer constant, since each one is calculated by taking the sum over $N_{in}$ inputs multiplied by the weights (and standard deviation of the sum of independent random variables scales as the square root of number of variables).

This initialization technique is called **uniform Kaiming initialization**. A few last notes on initialization methods:

- Kaiming often has a different constant in the numerator depending on what the target variance is, also there are uniform & normal variants of it (we'll only be using the uniform variant)
- **Xavier initialization** is the other well-known technique, and differs in that it uses $N_{in} + N_{out}$ in the denominator (this makes sense when also considering variance scaling of backward passes as well as forward passes - see the next dropdown for technical details)

<details>
<summary>Technical details (derivation of distribution)</summary>

The key intuition behind Kaiming initialisation (and others like it) is that we want the variance of our activations to be the same through all layers of the model when we initialize. Suppose $x$ and $y$ are activations from two adjacent layers, and $w$ are the weights connecting them (so we have $y_i = \sum_j w_{ij} x_j + b_i$, where $b$ is the bias). With $N_{x}$ as the number of neurons in layer $x$, we have:

$$
\begin{aligned}
\operatorname{Var}\left(y_i\right)=\sigma_x^2 & =\operatorname{Var}\left(\sum_j w_{i j} x_j\right) \\
& =\sum_j \operatorname{Var}\left(w_{i j} x_j\right) \quad \text { Inputs and weights are independent of each other } \\
& =\sum_j \operatorname{Var}\left(w_{i j}\right) \cdot \operatorname{Var}\left(x_j\right) \quad \text { Variance of product of independent RVs with zero mean is product of variances } \\
& = N_x \cdot \sigma_x^2 \cdot \operatorname{Var}\left(w_{i j}\right) \quad \text { Variance equal for all } N_x \text { neurons, call this value } \sigma_x^2
\end{aligned}
$$

For this to be the same as $\sigma_x^2$, we need $\operatorname{Var}(w_{ij}) = \frac{1}{N_x}$, so the standard deviation is $\frac{1}{\sqrt{N_x}}$.

This is not exactly the case for the Kaiming uniform distribution (which has variance $\frac{12}{(2 \sqrt{N_x})^2} = \frac{3}{N_x}$), and as far as I'm aware there's no principled reason why PyTorch does this. But the most important thing is that the variance scales as $O(1 / N_x)$, rather than what the exact scaling constant is.

There are other initializations with some theoretical justification. For instance, **Xavier initialization** has a uniform distribution in the interval:

$$
\bigg[-\frac{\sqrt{6}}{\sqrt{N_{in} + N_{out} + 1}}, \frac{\sqrt{6}}{\sqrt{N_{in} + N_{out} + 1}}\bigg]
$$

which is motivated by the idea of both keeping the variance of activations constant and keeping the ***gradients*** constant when we backpropagate.

However, you don't need to worry about any of this here, just implement Kaiming He uniform with a bound of $\frac{1}{\sqrt{N_{in}}}$!
</details>

### Exercise - implement `Linear`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to ~10 minutes on this exercise.
> ```

Remember, you should define the weights (and bias, if `bias=True`) in the `__init__` block. Also, make sure not to mix up `bias` (which is the boolean parameter to `__init__`) and `self.bias` (which should either be the actual bias tensor, or `None` if `bias` is false).

You should also fill in `forward` (which will multiply the input by the weight matrix and add the bias, if present).

Lastly, you should fill in `extra_repr` to give a string representation of the `Linear` module. There are no tests for this method, you should just make sure it's suitably informative (this will help when printing out your model later on).

In [50]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias=True):
        """
        A simple linear (technically, affine) transformation.

        The fields should be named `weight` and `bias` for compatibility with PyTorch.
        If `bias is False, set `self.bias` to None.
     
           """
        super().__init__()
        # so t.rand gies values in [0, 1) and we want [-1/sqrt(Nin), 1/sqrt(Nin)]
        # I need to first shift center to around 0, now we have [0.5, 0.5)
        # then I need to scale by multiplying by 2*(1/sqrt(Nin))
        self.weight = nn.Parameter((t.rand(out_features, in_features) - 0.5) * 2*(1/t.sqrt(t.tensor(in_features))))
        if bias:
            self.bias = nn.Parameter((t.rand(out_features) - 0.5) * 2*(1/t.sqrt(t.tensor(in_features))))
        else:
            self.bias = None
        
    def forward(self, x: Tensor) -> Tensor:
        """
        x: shape(*, in_features)
        Return: shape (*, out_features)
        """
        if self.bias is not None:
            return x @ self.weight.T + self.bias
        else:
            return x @ self.weight.T
    
    def extra_repr(self) -> str:
        return f"weight size={self.weight.size()}, bias={True if self.bias is not None else False}"

In [51]:
tests.test_linear_parameters(Linear, bias=False)
tests.test_linear_parameters(Linear, bias=True)
tests.test_linear_forward(Linear, bias=False)
tests.test_linear_forward(Linear, bias=True)

All tests in `test_linear_parameters` passed!
All tests in `test_linear_parameters` passed!
All tests in `test_linear_forward` passed!
All tests in `test_linear_forward` passed!


In [52]:
layer = Linear(2, 3, bias= True)

x = t.tensor([1.0, 2.0])   # shape (2,)
y = layer(x)

print("Output:", y)
print("Shape:", y.shape)
print(layer)

Output: tensor([-1.9107,  0.3210, -0.4739], grad_fn=<AddBackward0>)
Shape: torch.Size([3])
Linear(weight size=torch.Size([3, 2]), bias=True)


Notes from the provided solution:
- `sf` in `sf = 1 / sqrt(in_features)` stands for scale factor? we're defining how wide the random interval is.
- I need store key hyperparameters (e.g., `in_features`, `out_features`) as attributes in `__init__` for readability, debugging
- Here’s a clean single bullet point you can add for that part:
- To sample from a custom interval (e.g. ([-a, a])), transform `torch.rand()` by **centering and scaling**: e.g. `2*rand - 1` maps ([0,1)) → ([-1,1)), then multiply by a scale factor (`a`) to get ([-a, a]); equivalent forms like `(rand - 0.5) * 2 * a` follow the same “shift then scale” pattern
- I want to practice using `einsum` to implement operations like matrix multiplication (instead of default `@`) to build intuition for how tensor dimensions interact

### Flatten

Lastly, we've given you the `Flatten` module rather than including it as an exercise (because it's simple but quite finnicky to implement). This is a standardised way to rearrange our tensors so that they can be fed into a linear layer. It's a bit like `einops.rearrange`, but more specialised and less flexible (it flattens over some contiguous range of dimensions, rather than allowing for general reshape operations). By default we use `Flatten(start_dim=1, end_dim=-1)` which means flattening over the dimensions from `input.shape[1:]`, in other words over all except the batch dimension.

Make sure you understand what this module is doing before moving on.

In [59]:
class Flatten(nn.Module):
    def __init__(self, start_dim: int = 1, end_dim: int = -1) -> None:
        super().__init__()
        self.start_dim = start_dim
        self.end_dim = end_dim

    def forward(self, input: Tensor) -> Tensor:
        """
        Flatten out dimensions from start_dim to end_dim, inclusive of both.
        """
        shape = input.shape

        # Get start & end dims, handling negative indexing for end dim
        start_dim = self.start_dim
        end_dim = self.end_dim if self.end_dim >= 0 else len(shape) + self.end_dim

        # Get the shapes to the left / right of flattened dims, as well as size of flattened middle
        shape_left = shape[:start_dim]
        shape_right = shape[end_dim + 1:]
        shape_middle = t.prod(t.tensor(shape[start_dim : end_dim + 1])).item()

        return t.reshape(input, shape_left + (shape_middle,) + shape_right)
    
    def extra_repr(self) -> str:
        return ", ".join([f"{key}={getattr(self, key)}" for key in ["start_dim", "end_dim"]])

* convert multi-dimensional data (e.g. images) into `(batch, features)` so it can be fed into a linear layer
* Flatten = combine multiple dimensions into one long feature dimension (e.g. `(1, 28, 28) → 784`)
* Only works on a **contiguous range of dimensions** (a continuous block, not arbitrary dims)
* Default `Flatten(start_dim=1, end_dim=-1)` = flatten everything except the batch dimension
* keep batch dimension separate, flatten the rest: `(batch, features...) → (batch, flattened_features)`
* Similar to `einops.rearrange` but much more limited (only merges dims, doesn’t reorder)
* `t.prod` multiplies all elements in a tensor (used here to compute flattened size, e.g. `(3,4,5) → 60`)
* `(shape_middle,)` uses a comma to make a **1-element tuple** so it can be concatenated with other shape tuples

In [54]:
x = t. tensor([
    [[1, 2],
     [3, 4]],

     [[5, 6],
      [7, 8]]
])
x.shape

torch.Size([2, 2, 2])

In [60]:
flatten = Flatten()
print(flatten)

Flatten(start_dim=1, end_dim=-1)


In [61]:
x_flat = flatten(x)
x_flat, x_flat.shape

(tensor([[1, 2, 3, 4],
         [5, 6, 7, 8]]),
 torch.Size([2, 4]))

### Simple Multi-Layer Perceptron

Now, we can put together these two modules to create a neural network. We'll create one of the simplest networks which can be used to separate data that is non-linearly separable: a single linear layer, followed by a nonlinear function (ReLU), followed by another linear layer. This type of architecture (alternating linear layers and nonlinear functions) is often called a **multi-layer perceptron** (MLP).

The output of this network will have 10 dimensions, corresponding to the 10 classes of MNIST digits. We can then use the **softmax function** $x_i \to \frac{e^{x_i}}{\sum_i e^{x_i}}$ to turn these values into probabilities. However, it's common practice for the output of a neural network to be the values before we take softmax, rather than after. We call these pre-softmax values the **logits**.

<details>
<summary>Question - can you see what makes logits non-unique (i.e. why any given set of probabilities might correspond to several different possible sets of logits)?</summary>

Logits are **translation invariant**. If you add some constant $c$ to all logits $x_i$, then the new probabilities are:

$$
p_i' = \frac{e^{x_i + c}}{\sum_j e^{x_j + c}} = \frac{e^{x_i}}{\sum_j e^{x_j}} = p_i
$$

in other words, the probabilities don't change.

We can define **logprobs** as the log of the probabilities, i.e. $y_i = \log p_i$. Unlike logits, these are uniquely defined.

</details>

> to answer the question, I think we cannot recover logits from probabilities and different logits can produce the same probabilities after applying softmax, right? softmax doesn't really care about the logits values and cares more about the current logic value relevant to all logits values produced, right?

Notes:
* Softmax is translation invariant: adding the same constant to all logits doesn’t change probabilities
* This happens because the scaling factor cancels out in numerator and denominator
* logits are not unique: many shifted versions produce the same probabilities
* Softmax depends only on relative differences, not absolute values
* Log probabilities (`log(p)`) are unique, unlike logits


### Exercise - implement the simple MLP

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to ~20 minutes on this exercise.
> ```

The diagram below shows what your MLP should look like:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/mlp-mermaid.svg" width="170">

Please ask a TA (or message the Slack group) if any part of this diagram is unclear.

In [76]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = Flatten()
        self.linear1 = Linear(in_features=28*28, out_features=100)
        self.relu = ReLU()
        self.linear2 = Linear(in_features=100, out_features=10)
    
    def forward(self, x: Tensor) -> Tensor:
        x = self.flatten(x)
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        return x

In [77]:
tests.test_mlp_module(SimpleMLP)
tests.test_mlp_forward(SimpleMLP)

All tests in `test_mlp_module` passed!
All tests in `test_mlp_forward` passed!
